# ShiftGuard-X — NB94: Cent-Residual Mechanism Discrimination Gate

**Goal:** Falsify or advance one *narrow* hypothesis suggested by NB93: for monetary columns, a decoder-synchronous integer-cent residual stream with bit-exact fallback might beat an ordinary formula + packed-exception representation **after counting all bytes**. The ingredients (integer differencing, correlation-aware coding, and exceptions) are known prior art; even a positive outcome **does not establish invention or journal novelty**.

**Why this notebook:** NB93 achieved a 35.23% reduction on Bike Sharing and a 22.98% reduction on NYC Taxi against its strongest tested univariate / joint compression baselines. But NB93's modes themselves were known transformations, and Corra and Virtual already exploit taxi total_amount. This new notebook compares against the *actual nearest simple dependency baselines*, not only independent compressors.

**New candidate, not new invention:** `cent_delta` stores each source column using an independently calibrated, bitwise-verified codec; predicts target integer cents as the exact integer sum of independently rounded component cents; emits a mask for those target float64 words exactly reconstructible as an integer number of cents, compresses their signed integer residuals, and stores every remaining target word verbatim. Frame includes the schema, codec identifiers, lengths, and a checksum. Signed zero, NaNs with different payloads, infinities and non-cent values are preserved bitwise through fallback.

**Prior-art-equivalent controls:** NB93's raw target, formula+exceptions (raw/cents), tri-state exceptions, XOR residual. Treat the smallest held-out control as a conservative *post-test oracle*, **not** a deployable codec. Separately report the calibration-frozen selected method, and never use the held-out winner as the deployable choice.

**Gate declared before testing:** `cent_delta` must be calibration-selected and at least **5% smaller on NYC Taxi** than the smallest established NB93 mode. Bike is an exact-dependency negative/ceiling control: require no worse than **2% larger** than the best established mode on Bike (selection on Bike may remain with an established mode). Both datasets must decode bitwise. If NYC fails, reject this branch; if it passes, require independent fresh payment-family replication. If it passes, only consider further research after a direct Corra/C3/Virtual comparison and independent unseen datasets. A positive result does not imply an original algorithm.

**Input:** Public UCI Bike Sharing and January 2024 NYC TLC Yellow Taxi datasets. Fixed first 25% calibration, following 75% held out. Source/held-out SHA-256 must match the NB93 run supplied in this chat. If different, mark **INCOMPLETE** rather than compare mismatched experiments.

**Output:** `SGX_NB94_results.zip`; upload the ZIP after running all cells in a fresh Google Colab CPU runtime. No synthetic data substitute for missing downloads.


In [ ]:
# Install version-pinned dependencies in Colab; explicitly record any failures.
import sys, subprocess, importlib.util
required = {'zstandard':'0.25.0','pcodec':'1.0.3','pyarrow':'20.0.0','pandas':'2.2.2','numpy':'2.0.2'}
INSTALL = {}
for package, version in required.items():
    if importlib.util.find_spec(package) is not None:
        INSTALL[package] = 'already available'
    else:
        p = subprocess.run([sys.executable,'-m','pip','-q','install',f'{package}=={version}'],text=True,capture_output=True)
        INSTALL[package] = {'returncode':p.returncode,'tail':p.stderr[-350:]}
print(INSTALL)


In [ ]:
import os, io, csv, json, time, lzma, struct, hashlib, zipfile, urllib.request, platform, traceback
from pathlib import Path
import numpy as np, pandas as pd
import zstandard as zstd
try:
    import pcodec
    from pcodec import standalone as pcs
except Exception:
    pcodec=pcs=None
import pyarrow.parquet as pq
ROOT=Path('/content') if Path('/content').exists() else Path('/mnt/data')
OUT=ROOT/'SGX_NB94_outputs'; OUT.mkdir(exist_ok=True)
CACHE=ROOT/'SGX_NB94_source_cache'; CACHE.mkdir(exist_ok=True)
ERR=[]; DATA=[]; PROVENANCE=[]
UCI_URL='https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip'
NYC_URL='https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet'
MAX_TAXI_ROWS=120000

def download_exact(url, dest):
    if not dest.exists() or dest.stat().st_size == 0:
        req=urllib.request.Request(url,headers={'User-Agent':'ShiftGuardX-Research/93 (noncommercial academic benchmark)'})
        with urllib.request.urlopen(req,timeout=100) as src, open(dest,'wb') as dst:
            while True:
                b=src.read(1024*1024)
                if not b:break
                dst.write(b)
    b=dest.read_bytes()
    return hashlib.sha256(b).hexdigest(),len(b)

def load_bike():
    path=CACHE/'bike_sharing_uci275.zip'
    sha,sz=download_exact(UCI_URL,path)
    with zipfile.ZipFile(path) as z:
        member=next(s for s in z.namelist() if s.lower().endswith('hour.csv'))
        df=pd.read_csv(io.BytesIO(z.read(member)))
    names=['casual','registered','cnt']
    assert all(s in df.columns for s in names)
    m={s:np.ascontiguousarray(df[s].to_numpy(dtype='<f8')) for s in names}
    PROVENANCE.append({'family':'UCI-Bike-275','url':UCI_URL,'sha256':sha,'source_bytes':sz,'n_rows':len(df),'source_member':member,'transform':'Original integer-valued columns converted exactly to IEEE float64; not CSV-byte restoration'})
    return {'family':'UCI-Bike-275','sources':names[:-1], 'target':names[-1], 'values':m}

def load_taxi():
    path=CACHE/'yellow_tripdata_2024-01.parquet'
    sha,sz=download_exact(NYC_URL,path)
    pqfile=pq.ParquetFile(path)
    available={s.lower():s for s in pqfile.schema_arrow.names}
    required=['fare_amount','extra','mta_tax','tip_amount','tolls_amount','improvement_surcharge','congestion_surcharge','airport_fee','total_amount']
    missing=[c for c in required if c not in available]
    if missing:raise ValueError('Source Parquet missing required fields: '+str(missing))
    actual=[available[k] for k in required]
    # Bounded memory: only columns needed, and stop after first MAX_TAXI_ROWS.
    tables=[];nrows=0
    for batch in pqfile.iter_batches(batch_size=25000,columns=actual):
        if nrows>=MAX_TAXI_ROWS:break
        b=batch.to_pandas().iloc[:MAX_TAXI_ROWS-nrows]
        tables.append(b);nrows+=len(b)
    df=pd.concat(tables,ignore_index=True)
    m={canon:np.ascontiguousarray(pd.to_numeric(df[available[canon]],errors='coerce').to_numpy(dtype='<f8')) for canon in required}
    PROVENANCE.append({'family':'NYC-TLC-Yellow-2024-01','url':NYC_URL,'sha256':sha,'source_bytes':sz,'n_rows':len(df),'source_member':'first 120000 parquet rows; selected numeric columns','transform':'Parquet-decoded float64 bit patterns; null conversion to numpy NaN, not original parquet-file bytes'})
    return {'family':'NYC-TLC-Yellow-2024-01','sources':required[:-1], 'target':required[-1], 'values':m}

for label,fn in [('UCI-Bike-275',load_bike),('NYC-TLC-Yellow-2024-01',load_taxi)]:
    try:
        d=fn();DATA.append(d)
        print(label,'loaded rows:',len(d['values'][d['target']]))
    except Exception as e:
        ERR.append({'family':label,'stage':'data_loading','error':repr(e)})
        print(label,'INCOMPLETE:',repr(e))


In [ ]:
# Complete-file coder; every method carries its decoding metadata.
MAGIC=b'SG94'; C=zstd.ZstdCompressor(level=3);D=zstd.ZstdDecompressor()

def as_f8(arr):return np.asarray(arr,dtype='<f8').copy()
def fbits(arr):return as_f8(arr).view('<u8')
def as_u8(data):return np.frombuffer(data,dtype='<u8').copy()

def split8(b):
    x=np.frombuffer(b,dtype=np.uint8)
    if len(x)%8:raise ValueError('nonaligned byte stream')
    return x.reshape(-1,8).T.copy().tobytes()
def unsplit8(b):
    x=np.frombuffer(b,dtype=np.uint8)
    if len(x)%8:raise ValueError('nonaligned split stream')
    return x.reshape(8,-1).T.copy().tobytes()

def enc_bytes(raw,codec):
    if codec=='zstd':return C.compress(raw)
    if codec=='zstd_split':return C.compress(split8(raw))
    if codec=='lzma':return lzma.compress(raw,preset=1)
    raise ValueError(codec)
def dec_bytes(payload,codec):
    if codec=='zstd':return D.decompress(payload)
    if codec=='zstd_split':return unsplit8(D.decompress(payload))
    if codec=='lzma':return lzma.decompress(payload)
    raise ValueError(codec)

def enc_column(v,method):
    if method=='pcodec':
        if pcs is None:raise RuntimeError('pcodec unavailable')
        return bytes(pcs.simple_compress(as_f8(v),pcodec.ChunkConfig()))
    return enc_bytes(as_f8(v).tobytes(),method)
def dec_column(p,method,n):
    if method=='pcodec':
        if pcs is None:raise RuntimeError('pcodec unavailable')
        v=as_f8(pcs.simple_decompress(p))
    else:
        raw=dec_bytes(p,method)
        if len(raw)!=8*n:raise ValueError('wrong decoded byte count')
        v=np.frombuffer(raw,dtype='<f8').copy()
    if len(v)!=n:raise ValueError('wrong decoded element count')
    return v

METHODS=['zstd','zstd_split','lzma']+(['pcodec'] if pcs is not None else [])
def frozen_codec(calib):
    outcomes=[]
    for method in METHODS:
        try:
            t=time.perf_counter();p=enc_column(calib,method)
            assert dec_column(p,method,len(calib)).tobytes()==as_f8(calib).tobytes()
            outcomes.append((len(p),method,time.perf_counter()-t))
        except Exception as e:ERR.append({'stage':'codec_trial','codec':method,'error':repr(e)})
    if not outcomes:raise RuntimeError('no bit-exact compressor available')
    return min(outcomes)[1],outcomes

def predictors(values,sources):
    n=len(values[sources[0]])
    with np.errstate(all='ignore'):
        raw=np.zeros(n,dtype='<f8')
        for name in sources:raw=np.add(raw,values[name],dtype='<f8')
        cents=np.zeros(n,dtype=np.int64)
        valid=np.ones(n,dtype=bool)
        for name in sources:
            v=as_f8(values[name])
            okay=np.isfinite(v)&(np.abs(v)<1e12)
            valid &= okay
            vv=np.where(okay,v,0)
            cents+=np.rint(vv*100.0).astype(np.int64)
        cents_pred=cents.astype('<f8')/100.0
        cents_pred[~valid]=np.nan
    return raw,cents_pred

def masks(values,sources,target,mode):
    truth=fbits(values[target]); a,b=predictors(values,sources)
    a_bits=fbits(a);b_bits=fbits(b)
    if mode=='mask_raw':
        code=np.where(truth==a_bits,0,2).astype(np.uint8)
    elif mode=='mask_cents':
        code=np.where(truth==b_bits,0,2).astype(np.uint8)
    elif mode=='mask_tri':
        code=np.where(truth==a_bits,0,np.where(truth==b_bits,1,2)).astype(np.uint8)
    else:raise ValueError(mode)
    return code

def pack_mode(code,tri):
    if tri:
        n=len(code);pad=(-n)%4
        if pad:code=np.pad(code,(0,pad),constant_values=3)
        x=code.reshape(-1,4)
        return ((x[:,0]<<6)|(x[:,1]<<4)|(x[:,2]<<2)|x[:,3]).astype('u1').tobytes()
    return np.packbits(code==2,bitorder='big').tobytes()

def unpack_mode(raw,n,tri):
    if tri:
        x=np.frombuffer(raw,dtype='u1');out=np.empty(len(x)*4,dtype='u1')
        out[0::4]=x>>6;out[1::4]=(x>>4)&3;out[2::4]=(x>>2)&3;out[3::4]=x&3
        return out[:n].copy()
    return (np.unpackbits(np.frombuffer(raw,dtype='u1'),bitorder='big')[:n]*2).astype('u1')

def table_hash(values,names):
    h=hashlib.sha256()
    for name in names:h.update(as_f8(values[name]).tobytes())
    return h.hexdigest()

def frame_pack(meta,chunks):
    assert len(meta['chunks'])==len(chunks)
    m=json.dumps(meta,sort_keys=True,separators=(',',':'),allow_nan=False).encode('utf8')
    return MAGIC+struct.pack('<I',len(m))+m+b''.join(chunks)

def frame_unpack(p):
    if len(p)<8 or p[:4]!=MAGIC:raise ValueError('frame truncated or invalid')
    mlen=struct.unpack_from('<I',p,4)[0]
    if mlen>1024*1024 or len(p)<8+mlen:raise ValueError('truncated metadata')
    meta=json.loads(p[8:8+mlen]);pos=8+mlen;chunks=[]
    for info in meta['chunks']:
        length=int(info['nbytes'])
        if length<0 or pos+length>len(p):raise ValueError('truncated chunk')
        chunks.append(p[pos:pos+length]);pos+=length
    if pos!=len(p):raise ValueError('trailing frame bytes')
    return meta,chunks

def encode(values,sources,target,mode,frozen_methods):
    n=len(values[target]); names=sources+[target];parts=[];tags=[]
    def add(kind,p):
        tags.append({'kind':kind,'nbytes':len(p)});parts.append(p)
    for src in sources:add('col:'+src,enc_column(values[src],frozen_methods[src]))
    targetv=as_f8(values[target]);truth=targetv.view('<u8')
    if mode=='raw':
        add('target',enc_column(targetv,frozen_methods[target]))
    elif mode in ('mask_raw','mask_cents','mask_tri'):
        code=masks(values,sources,target,mode)
        add('mask',C.compress(pack_mode(code,mode=='mask_tri')))
        add('exceptions',C.compress(truth[code==2].astype('<u8').tobytes()))
    elif mode=='xor_raw':
        pred,_=predictors(values,sources)
        add('xor',C.compress(split8((truth^fbits(pred)).astype('<u8').tobytes())))
    else:raise ValueError(mode)
    meta={'schema':'SG93-v1','n':n,'sources':sources,'target':target,'mode':mode,'frozen_methods':frozen_methods,'raw_sha256':table_hash(values,names),'chunks':tags}
    return frame_pack(meta,parts)

def decode(frame):
    meta,parts=frame_unpack(frame);n=int(meta['n']);sources=meta['sources'];target=meta['target'];mode=meta['mode'];methods=meta['frozen_methods']
    values={s:dec_column(p,methods[s],n) for s,p in zip(sources,parts[:len(sources)])}
    extra=parts[len(sources):]
    if mode=='raw':
        assert len(extra)==1
        values[target]=dec_column(extra[0],methods[target],n)
    elif mode in ('mask_raw','mask_cents','mask_tri'):
        if len(extra)!=2:raise ValueError('bad masks')
        tri=(mode=='mask_tri');mraw=D.decompress(extra[0]);expected=(n+3)//4 if tri else (n+7)//8
        if len(mraw)!=expected:raise ValueError('bad mask byte count')
        code=unpack_mode(mraw,n,tri)
        if np.any(code>2) or (not tri and np.any(code==1)):raise ValueError('invalid mask code')
        pred_a,pred_b=predictors(values,sources)
        words=fbits(pred_a if mode!='mask_cents' else pred_b)
        if tri:words=np.where(code==1,fbits(pred_b),words)
        exceptions=as_u8(D.decompress(extra[1]))
        if len(exceptions)!=int(np.sum(code==2)):raise ValueError('wrong exception count')
        words[code==2]=exceptions
        values[target]=words.view('<f8').copy()
    elif mode=='xor_raw':
        if len(extra)!=1:raise ValueError('bad xor format')
        words=as_u8(unsplit8(D.decompress(extra[0])))
        if len(words)!=n:raise ValueError('bad xor values')
        a,_=predictors(values,sources)
        values[target]=(words^fbits(a)).view('<f8').copy()
    else:raise ValueError('unrecognized mode')
    if table_hash(values,sources+[target])!=meta['raw_sha256']:raise ValueError('checksum mismatch')
    return values


In [ ]:
# Local algorithmic smoke test only. NEVER call these data "real dataset results".
words=np.array([0x0000000000000000,0x8000000000000000,0x7ff0000000000000,0xfff0000000000000,0x7ff8000000000001,0x7ff80000000000ab,0x0000000000000001,0x7fefffffffffffff],dtype='<u8')
odd=words.view('<f8').copy(); parents={'a':odd,'b':np.array([1.,-2.,3.,4.,5.,6.,7.,8.],dtype='<f8')}
y=odd.copy();values={**parents,'y':y}
for mode in ['raw','mask_raw','mask_cents','mask_tri','xor_raw']:
    methods={k:'zstd' for k in values}
    buf=encode(values,['a','b'],'y',mode,methods)
    restored=decode(buf)
    assert all(fbits(restored[k]).tobytes()==fbits(values[k]).tobytes() for k in values)
    try:decode(buf[:-1]);raise AssertionError('truncation was accepted')
    except (ValueError,zstd.ZstdError,lzma.LZMAError):pass
print('PASS: all five modes, signed zeros, payload-distinct NaNs, infinities, subnormal, truncated frames')


In [ ]:
# New candidate mode. Local correctness tests do NOT represent real dataset results.
# All arithmetic must be decoder-reconstructible using integer cents and the exact stored source words.

def cent_integers(v):
    v=as_f8(v)
    valid=np.isfinite(v) & (np.abs(v)<1e12)
    protected=np.where(valid,v,0.0)
    result=np.rint(protected*100.0).astype('<i8')
    canonical=valid & (fbits(result.astype('<f8')/100.0)==fbits(v))
    return result,canonical,valid

def cent_parent_sum(values,parents):
    n=len(values[parents[0]])
    acc=np.zeros(n,dtype='<i8');valid=np.ones(n,dtype=bool)
    for parent in parents:
        cents,_,okay=cent_integers(values[parent])
        valid &= okay
        acc += cents
    return acc,valid

def cent_delta_encode(values,parents,target,methods):
    n=len(values[target]);names=parents+[target]
    pp=[];infos=[]
    def append(kind,blob):
        pp.append(blob);infos.append({'kind':kind,'nbytes':len(blob)})
    for col in parents:append('col:'+col,enc_column(values[col],methods[col]))
    target_cents,target_canonical,_=cent_integers(values[target]);pred,valid=cent_parent_sum(values,parents)
    use=target_canonical&valid
    # Target words not exactly reproducible from cents, including -0 and NaN payloads, are stored verbatim.
    mismatch=np.packbits(~use,bitorder='big').tobytes()
    append('exception_mask',C.compress(mismatch))
    residual=(target_cents[use]-pred[use]).astype('<i8')
    append('cent_residuals',C.compress(split8(residual.tobytes())))
    append('original_exception_words',C.compress(fbits(values[target])[~use].astype('<u8').tobytes()))
    meta={'schema':'SG94-cent-delta-v1','n':n,'sources':parents,'target':target,
          'methods':methods,'raw_sha256':table_hash(values,names),'chunks':infos}
    return frame_pack(meta,pp),{'canonical_fraction':float(np.mean(use)),'exceptions':int((~use).sum()),
                  'n_residuals':int(use.sum()),'nonzero_cent_residuals':int(np.count_nonzero(residual)),
                  'payload_bytes':{k['kind']:k['nbytes'] for k in infos}}

def cent_delta_decode(blob):
    meta,parts=frame_unpack(blob)
    if meta.get('schema')!='SG94-cent-delta-v1':raise ValueError('wrong cent-delta schema')
    parents=meta['sources'];target=meta['target'];n=int(meta['n']);methods=meta['methods']
    if len(parts)!=len(parents)+3:raise ValueError('wrong chunk count')
    values={col:dec_column(payload,methods[col],n) for col,payload in zip(parents,parts[:len(parents)])}
    maskpayload,residualpayload,exceptionpayload=parts[len(parents):]
    maskbytes=D.decompress(maskpayload)
    if len(maskbytes)!=(n+7)//8:raise ValueError('wrong cent mask size')
    use=~np.unpackbits(np.frombuffer(maskbytes,dtype='u1'),bitorder='big')[:n].astype(bool)
    residualbytes=unsplit8(D.decompress(residualpayload))
    if len(residualbytes)!=8*int(use.sum()):raise ValueError('wrong residual stream length')
    residual=np.frombuffer(residualbytes,dtype='<i8')
    exceptionbytes=D.decompress(exceptionpayload)
    if len(exceptionbytes)!=8*int((~use).sum()):raise ValueError('wrong exception length')
    exceptions=np.frombuffer(exceptionbytes,dtype='<u8')
    pred,valid=cent_parent_sum(values,parents)
    if np.any(use&(~valid)):raise ValueError('invalid cent prediction')
    words=np.empty(n,dtype='<u8')
    words[~use]=exceptions
    vals=(pred[use]+residual).astype('<f8')/100.0
    words[use]=fbits(vals)
    values[target]=words.view('<f8').copy()
    if table_hash(values,parents+[target])!=meta['raw_sha256']:raise ValueError('bitwise checksum mismatch')
    return values

# Code-level controls: exact cents, fractional-cent exception, different NaN payloads, -0, infinity.
sample={
 'a':np.array([1.11,2.25,-0.0,0.3,float('inf'),1.4,1e-8,-10.0],dtype='<f8'),
 'b':np.array([2.22,3.50,0.0,0.2,7.,1.1,-1e-8,9.0],dtype='<f8')}
sample['total']=np.array([3.33,5.75,-0.0,0.5,4.5,2.503,0.0, -1.0],dtype='<f8')
words=fbits(sample['total']).copy();words[4]=np.uint64(0x7ff80000000000ab)
sample['total']=words.view('<f8').copy()
my_methods={k:'zstd' for k in sample}
blob,_=cent_delta_encode(sample,['a','b'],'total',my_methods)
dec=cent_delta_decode(blob)
assert all(fbits(sample[k]).tobytes()==fbits(dec[k]).tobytes() for k in sample)
try:
    cent_delta_decode(blob[:-2]);raise AssertionError('Truncation incorrectly accepted')
except (ValueError,zstd.ZstdError,lzma.LZMAError):pass
print('PASS: bit-exact cent-delta local smoke, NaN payloads, signed zero, subcent values, truncated frames')


In [ ]:
# Real-dataset *matched-comparison* evaluation. No imputation or synthetic substitute.
import collections
EXPECTED_SOURCE_HASHES={
 'UCI-Bike-275':'b70182d0d0508e9abbb79306ce5c0cec34869000f8220175ac83d11dbe845401',
 'NYC-TLC-Yellow-2024-01':'c4d59da7bbc8abaeeeb1727947ee93d9891a71acb42854bd80db1571b2030510'
}
EXPECTED_HOLD_HASHES={
 'UCI-Bike-275':'710e66352e1d341f53797dc9535f6436589ee99bb7765f979209ee474edf1693',
 'NYC-TLC-Yellow-2024-01':'3d0fc4346b1b4d3457aaa1e9941fc2729c67a76ca405bc1fb6829d7e22f8f651'
}
MODES=['raw','mask_raw','mask_cents','mask_tri','xor_raw']
RESULTS=[];MODES_DATA=[];ERRORS=list(ERR);CONTROLS=[]
source_status={p['family']:(p['sha256']==EXPECTED_SOURCE_HASHES.get(p['family'])) for p in PROVENANCE}
for d in DATA:
    fam=d['family'];sources=d['sources'];target=d['target'];names=sources+[target];data=d['values']
    n=len(data[target]);cut=max(50,int(n*.25))
    cal={k:v[:cut] for k,v in data.items()};hold={k:v[cut:] for k,v in data.items()}
    sha=table_hash(hold,names)
    if not source_status.get(fam,False) or sha!=EXPECTED_HOLD_HASHES.get(fam):
        ERRORS.append({'family':fam,'stage':'provenance','error':'source or held-out hash does not match NB93; no comparison'})
        continue
    try:
        if pcs is None: raise RuntimeError('Native PCodec unavailable; cannot complete matched comparison')
        t0=time.perf_counter();methods={col:frozen_codec(cal[col])[0] for col in names}
        training={}
        for mode in MODES:
            p=encode(cal,sources,target,mode,methods)
            assert all(fbits(decode(p)[k]).tobytes()==fbits(cal[k]).tobytes() for k in names)
            training[mode]=len(p)
        novel_train,_=cent_delta_encode(cal,sources,target,methods)
        assert all(fbits(cent_delta_decode(novel_train)[k]).tobytes()==fbits(cal[k]).tobytes() for k in names)
        training['cent_delta']=len(novel_train)
        selected=min(sorted(training),key=lambda x:training[x]);cal_s=time.perf_counter()-t0
        sizes={};details={}
        for mode in MODES:
            start=time.perf_counter();p=encode(hold,sources,target,mode,methods);enc_s=time.perf_counter()-start
            start=time.perf_counter();restored=decode(p);dec_s=time.perf_counter()-start
            assert all(fbits(restored[k]).tobytes()==fbits(hold[k]).tobytes() for k in names)
            sizes[mode]=len(p)
            MODES_DATA.append({'family':fam,'mode':mode,'training_bytes':training[mode],'heldout_bytes':len(p),'selected_before_holdout':selected==mode,'encode_s':enc_s,'decode_s':dec_s,'bits_verified':True})
        start=time.perf_counter();p,details=cent_delta_encode(hold,sources,target,methods);enc_s=time.perf_counter()-start
        start=time.perf_counter();restored=cent_delta_decode(p);dec_s=time.perf_counter()-start
        assert all(fbits(restored[k]).tobytes()==fbits(hold[k]).tobytes() for k in names)
        sizes['cent_delta']=len(p)
        MODES_DATA.append({'family':fam,'mode':'cent_delta','training_bytes':training['cent_delta'],'heldout_bytes':len(p),'selected_before_holdout':selected=='cent_delta','encode_s':enc_s,'decode_s':dec_s,'bits_verified':True})
        # Native PCodec standalone comparator must decode exactly, as in NB93.
        native_frame=encode(hold,sources,target,'raw',{k:'pcodec' for k in names})
        native_decoded=decode(native_frame)
        assert all(fbits(native_decoded[k]).tobytes()==fbits(hold[k]).tobytes() for k in names)
        native_bytes=len(native_frame)
        closest=min(sizes[k] for k in MODES)
        ref_mode=min(MODES,key=lambda x:sizes[x])
        margin=100*(closest-sizes['cent_delta'])/closest
        RESULTS.append({'family':fam,'calibration_rows':cut,'heldout_rows':n-cut,'source_hash_matches_NB93':True,'heldout_hash_matches_NB93':True,
          'calibration_selected_mode':selected,'cent_delta_selected':selected=='cent_delta','cent_delta_bytes':sizes['cent_delta'],
          'best_known_mode':ref_mode,'best_known_mode_bytes':closest,'matched_native_pcodec_bytes':native_bytes,'cent_delta_margin_pct':margin,
          'cent_delta_exceptions':details['exceptions'],'cent_delta_canonical_fraction':details['canonical_fraction'],
          'cent_delta_nonzero_residuals':details['nonzero_cent_residuals'],'calibration_seconds':cal_s,
          'cent_delta_passes_five_percent':selected=='cent_delta' and margin>=5.0,
          'heldout_sha256':sha})
        CONTROLS.append({'family':fam,**details['payload_bytes']})
        print(fam,'selected',selected,'cent_delta',len(p),'nearest known',closest,'margin %',round(margin,3))
    except Exception as e:
        ERRORS.append({'family':fam,'stage':'evaluation','error':repr(e)})
        print(fam,'ERROR',repr(e))


In [ ]:
# Freeze decision, write auditable outputs, then ZIP.
def write_csv(name,rows):
    fields=list(dict.fromkeys(k for r in rows for k in r)) if rows else ['status']
    with (OUT/name).open('w',newline='',encoding='utf8') as f:
        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows(rows if rows else [{'status':'NO_ROWS'}])
for name,rows in [('source_provenance.csv',PROVENANCE),('matched_mode_sizes.csv',MODES_DATA),('family_decision.csv',RESULTS),('cent_delta_payload_breakdown.csv',CONTROLS),('run_issues.csv',ERRORS)]:write_csv(name,rows)
complete=(len(RESULTS)==2 and pcs is not None and not ERRORS and not ERR and all(r['source_hash_matches_NB93'] and r['heldout_hash_matches_NB93'] for r in RESULTS))
by_family={r['family']:r for r in RESULTS}
taxi=by_family.get('NYC-TLC-Yellow-2024-01',{})
bike=by_family.get('UCI-Bike-275',{})
passes=bool(complete and taxi.get('cent_delta_passes_five_percent',False) and bike.get('cent_delta_margin_pct',-100)>=-2.0)
report={'notebook':'NB94','status':'COMPLETE' if complete else 'INCOMPLETE',
  'datasets_evaluated':len(RESULTS),'experimental_hypothesis_pass':bool(passes),
  'targeted_gate':'NYC frozen cent_delta >=5% vs best existing dependency mode, Bike regression <=2%; then independently replicate',
  'algorithmic_novelty_demonstrated':False,'journal_ready':False,
  'requires_direct_Corra_C3_Virtual_comparison':True,
  'decision':'CONTINUE_ONLY_TO_EXTERNAL_PRIOR_ART_AND_BLIND_CONFIRMATION' if passes else ('REJECT_CENT_DELTA_SIZE_HYPOTHESIS' if complete else 'REPAIR_EXPERIMENT'),
  'prior_art_caveat':'Arithmetic residuals, functional dependencies, exception fallback and bit-exact floats are established; even positive size outcomes do not constitute a novel codec.',
  'competing_known_modes_are_post_test_oracle_only':True,
  'data_caveat':'Restores parsed IEEE-754 streams, not original UCI CSV or NYC Parquet bytes'}
(OUT/'decision.json').write_text(json.dumps(report,indent=2))
(OUT/'environment.json').write_text(json.dumps({'python':sys.version,'platform':platform.platform(),'numpy':np.__version__,'pcodec_available':pcs is not None,'package_installs':INSTALL},indent=2,default=str))
final=ROOT/'SGX_NB94_results.zip'
with zipfile.ZipFile(final,'w',zipfile.ZIP_DEFLATED) as z:
    for f in sorted(OUT.iterdir()):
        if f.is_file():z.write(f,arcname=f.name)
print(json.dumps(report,indent=2))
print('Output:',final)
try:
    from google.colab import files
    files.download(str(final))
except ImportError:pass
